# Accountability and Governance

## Overview
Accountability answers a simple question: when an AI system causes harm, who is responsible and how is it put right? **Governance** is the set of policies, roles, and records that make accountability real. This notebook turns the major frameworks into runnable checklists and data structures, using only the Python standard library.

---

## 1. AI Governance Frameworks
A **governance framework** is a structured set of rules, roles, and processes for managing the risks of an AI system across its lifecycle. The dominant frameworks are the EU AI Act (law), the NIST AI Risk Management Framework (voluntary US guidance), and ISO/IEC 42001 (an international management-system standard).

## 2. The EU AI Act
The EU AI Act classifies systems by risk and regulates them accordingly:
- **Unacceptable risk**: prohibited outright (for example government social scoring).
- **High risk**: allowed but heavily regulated (for example AI in hiring, credit, or medical devices), with obligations for risk management, data quality, logging, human oversight, and documentation.
- **Limited risk**: transparency duties, such as disclosing that a user is interacting with a chatbot.
- **Minimal risk**: largely unregulated (for example a spam filter).

## 3. The NIST AI Risk Management Framework
The NIST AI RMF is built on four functions that run continuously: **Govern**, **Map**, **Measure**, and **Manage**. A common way to prioritize the risks found in the Map stage is a simple risk score:

$$\text{risk score} = \text{likelihood} \times \text{impact}$$

## 4. Standards, Registry, and Redress
Complementary standards include **ISO/IEC 42001** (an AI management system), the **OECD AI Principles**, and the **UNESCO Recommendation on the Ethics of AI**. In practice governance also needs a **model registry** with versioning, **audit trails** that log every consequential decision, and **redress mechanisms** so people affected by a decision can contest it.

---


In [1]:
# == EU AI Act: rule-based risk classifier =====================
# Maps a use case described by simple boolean attributes to a risk tier.

def classify_eu_ai_act_risk(attrs):
    """Return the EU AI Act risk tier for a use case.
    attrs is a dict of booleans describing the system."""
    if attrs.get('social_scoring') or attrs.get('manipulative') or attrs.get('realtime_biometric_id_public'):
        return 'Unacceptable (prohibited)'
    high_risk_domains = (
        attrs.get('hiring') or attrs.get('credit_scoring') or
        attrs.get('medical_device') or attrs.get('law_enforcement') or
        attrs.get('critical_infrastructure') or attrs.get('education_access')
    )
    if high_risk_domains:
        return 'High risk (strict obligations)'
    if attrs.get('chatbot') or attrs.get('generates_synthetic_media'):
        return 'Limited risk (transparency duties)'
    return 'Minimal risk'

examples = {
    'Resume screening tool': {'hiring': True},
    'Customer support chatbot': {'chatbot': True},
    'Government social scoring': {'social_scoring': True},
    'Movie recommender': {},
}
for name, a in examples.items():
    print(f'{name:30s} -> {classify_eu_ai_act_risk(a)}')


Resume screening tool          -> High risk (strict obligations)
Customer support chatbot       -> Limited risk (transparency duties)
Government social scoring      -> Unacceptable (prohibited)
Movie recommender              -> Minimal risk


In [2]:
# == Model card and append-only audit log ======================
from dataclasses import dataclass, field, asdict
from typing import List

@dataclass
class ModelCard:
    name: str
    version: str
    intended_use: str
    training_data: str
    metrics: dict
    known_limitations: str
    risk_tier: str

@dataclass
class AuditLog:
    # Timestamps are passed in as arguments so the log stays deterministic.
    entries: List[dict] = field(default_factory=list)
    def record(self, timestamp, actor, action, detail):
        self.entries.append({'timestamp': timestamp, 'actor': actor,
                             'action': action, 'detail': detail})

card = ModelCard(
    name='loan_approval', version='1.2.0',
    intended_use='Assist (not replace) human loan officers',
    training_data='Internal applications 2018-2023, reweighted for balance',
    metrics={'auc': 0.86, 'demographic_parity_ratio': 0.91},
    known_limitations='Underperforms on thin-file applicants',
    risk_tier=classify_eu_ai_act_risk({'credit_scoring': True}),
)

log = AuditLog()
log.record('2026-01-10T09:00:00Z', 'data_team', 'dataset_approved', 'v3 reweighted')
log.record('2026-01-12T14:30:00Z', 'ml_team', 'model_registered', 'loan_approval 1.2.0')
log.record('2026-01-15T11:05:00Z', 'risk_team', 'human_oversight_signoff', 'High-risk review passed')

print('Model card:')
for k, v in asdict(card).items():
    print(f'  {k}: {v}')
print('\nAudit trail:')
for e in log.entries:
    print(f"  {e['timestamp']} | {e['actor']:10s} | {e['action']}")


Model card:
  name: loan_approval
  version: 1.2.0
  intended_use: Assist (not replace) human loan officers
  training_data: Internal applications 2018-2023, reweighted for balance
  metrics: {'auc': 0.86, 'demographic_parity_ratio': 0.91}
  known_limitations: Underperforms on thin-file applicants
  risk_tier: High risk (strict obligations)

Audit trail:
  2026-01-10T09:00:00Z | data_team  | dataset_approved
  2026-01-12T14:30:00Z | ml_team    | model_registered
  2026-01-15T11:05:00Z | risk_team  | human_oversight_signoff


In [3]:
# == NIST AI RMF checklist generator ===========================
NIST_RMF = {
    'Govern':  ['Define AI risk policies and accountable owners',
                'Set up a model inventory and review board'],
    'Map':     ['Document context, stakeholders, and intended use',
                'Identify potential harms and impacted groups'],
    'Measure': ['Quantify risks with metrics (accuracy, fairness, drift)',
                'Track risk score = likelihood x impact'],
    'Manage':  ['Prioritize and mitigate the highest risks',
                'Plan monitoring, incident response, and redress'],
}

def print_rmf_checklist(framework):
    for function, actions in framework.items():
        print(f'[{function}]')
        for a in actions:
            print(f'  [ ] {a}')

print_rmf_checklist(NIST_RMF)


[Govern]
  [ ] Define AI risk policies and accountable owners
  [ ] Set up a model inventory and review board
[Map]
  [ ] Document context, stakeholders, and intended use
  [ ] Identify potential harms and impacted groups
[Measure]
  [ ] Quantify risks with metrics (accuracy, fairness, drift)
  [ ] Track risk score = likelihood x impact
[Manage]
  [ ] Prioritize and mitigate the highest risks
  [ ] Plan monitoring, incident response, and redress


In [4]:
# == Minimal model registry with semantic versioning ===========
# Semantic version: MAJOR.MINOR.PATCH
#   MAJOR = breaking change, MINOR = new capability, PATCH = fix

registry = {}

def register_model(name, version, stage, card):
    registry.setdefault(name, {})[version] = {'stage': stage, 'card': card}

def promote(name, version, stage):
    registry[name][version]['stage'] = stage

register_model('loan_approval', '1.1.0', 'archived', None)
register_model('loan_approval', '1.2.0', 'staging', card)
promote('loan_approval', '1.2.0', 'production')

for name, versions in registry.items():
    print(name)
    for v, meta in sorted(versions.items()):
        print(f'  {v}: {meta["stage"]}')


loan_approval
  1.1.0: archived
  1.2.0: production


## Additional Learning Resources

### Frameworks & Standards
- NIST AI Risk Management Framework (https://www.nist.gov/itl/ai-risk-management-framework)
- EU AI Act (Regulation 2024/1689) (https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:32024R1689)
- OECD AI Principles (https://www.oecd.org/digital/artificial-intelligence/)
- ISO/IEC 42001 AI Management System (https://www.iso.org/standard/81230.html)
- UNESCO Recommendation on the Ethics of AI (https://unesdoc.unesco.org/ark:/48223/pf0000381137)

### Guides
- NIST AI RMF Playbook (https://www.nist.gov/itl/ai-risk-management-framework)

### Tools & Libraries
- Credo AI (https://credo.ai/)
- Responsible AI Institute (https://www.responsible.ai/)
